In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import time
from selenium import webdriver

In [2]:
teams = pd.read_csv(os.getcwd().replace('scripts', 'data') + '/teams.csv')
matches = pd.read_csv(os.getcwd().replace('scripts', 'data') + '/all_matches.csv')

In [3]:
base = 'https://ftc-events.firstinspires.org/2022/team/'

In [4]:
teams

,Team Number,Team Name
0,5951,Mustangs
1,8990,Mile High Academy Mustangs
2,11020,SKI SKY
3,15931,Tin Heads
4,16899,CPUsaders_T
...,...,...
5908,23150,Dragon
5909,23152,飞虎创学队
5910,23153,梦之队
5911,23155,37中队


In [5]:
matches.head(10)

,Link,Name,Location,Start,End
0,USARLCMP,Adventist Robotics League Championship,USA,2023-05-07 00:00:00,2023-05-07 00:00:00
1,USMIAAOS,Girls R GRAYT All Girls Event,USA,2023-05-05 00:00:00,2023-05-06 00:00:00
2,USMIGIOS,Gibraltar FTC Spring Showdown,USA,2023-05-05 00:00:00,2023-05-06 00:00:00
3,USPAPHOS,PA FTC RoboJawn,USA,2023-05-06 00:00:00,2023-05-06 00:00:00
4,FRLEOS,FR Le Défi Robotique – Lyon 2023,France,2023-05-13 00:00:00,2023-05-13 00:00:00
5,USMITCOS2,MI TC West Remote Event,USA,2023-05-07 00:00:00,2023-05-13 00:00:00
6,USMIKIOS,Kingsford Spring Event,USA,2023-05-12 00:00:00,2023-05-13 00:00:00
7,USMIRCOS,Reed City Spring Event,USA,2023-05-12 00:00:00,2023-05-13 00:00:00
8,USMIJAOS,Jackson FTC Spring Event,USA,2023-05-12 00:00:00,2023-05-13 00:00:00
9,USORROOS,OR Spring Showcase,USA,2023-05-13 00:00:00,2023-05-13 00:00:00


In [6]:
url = 'http://ftc-api.firstinspires.org/v2.0/2022/matches/'
headers = {'Authorization': 'Basic szj2024:F370ADBF-DC4B-4364-807E-F56673B369BC'}

In [7]:
r = requests.get(url + matches.iat[1,0], auth=('szj2024', 'F370ADBF-DC4B-4364-807E-F56673B369BC'))

In [8]:
r.json()

{'matches': [{'actualStartTime': '2023-05-06T10:58:07.074',
   'description': 'Qualification 1',
   'tournamentLevel': 'QUALIFICATION',
   'series': 0,
   'matchNumber': 1,
   'scoreRedFinal': 37,
   'scoreRedFoul': 0,
   'scoreRedAuto': 0,
   'scoreBlueFinal': 39,
   'scoreBlueFoul': 0,
   'scoreBlueAuto': 4,
   'postResultTime': '2023-05-06T11:03:25.453',
   'teams': [{'teamNumber': 22373,
     'station': 'Blue2',
     'dq': False,
     'onField': True},
    {'teamNumber': 16523, 'station': 'Blue1', 'dq': False, 'onField': True},
    {'teamNumber': 15353, 'station': 'Red2', 'dq': False, 'onField': True},
    {'teamNumber': 7065, 'station': 'Red1', 'dq': False, 'onField': True}],
   'modifiedOn': '2023-05-06T21:33:31.418'},
  {'actualStartTime': '2023-05-06T11:05:08.389',
   'description': 'Qualification 2',
   'tournamentLevel': 'QUALIFICATION',
   'series': 0,
   'matchNumber': 2,
   'scoreRedFinal': 75,
   'scoreRedFoul': 30,
   'scoreRedAuto': 20,
   'scoreBlueFinal': 133,
   'sco

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FTC Team Statistics Calculator
@author: owsorber

OPR = Offensive Power Rating
CCWM = Calculated Contribution to Winning Margin
"""

import numpy # has matrix calculations

class Alliance:
    def __init__(self, team1, team2, score, auto, col):
        self.team1 = team1
        self.team2 = team2
        self.score = score
        self.auto = auto
        self.col = col
        if (self.auto > self.score):
            raise ValueError("Autonomous score cannot be higher than Overall score.")
    
    def __str__(self):
        return self.col + " Alliance: " + str(self.team1) + ", " + (self.team2)

class Match:
    def __init__(self, num, redAlliance, blueAlliance):
        self.num = num
        self.redAlliance = redAlliance
        self.blueAlliance = blueAlliance
    
    def __str__(self):
        return "Match # " + str(self.num) + ": (" + str(self.redAlliance.team1.num) + " & " + str(self.redAlliance.team2.num) + ") " + str(self.redAlliance.score) + " - " + str(self.blueAlliance.score) + " (" + str(self.blueAlliance.team1.num) + " & " +  str(self.blueAlliance.team2.num) + ")"


""" 
Loads the teams of an FTC event into a dictionary from an external txt file.
The dictionary maps a team number to a team name.
"""
def loadTeams(filename):
    teams = {}
    
    f = open(filename, 'r')
    
    for line in f:
#         print(line)
        line_data = line.split(', ')
#         print(line_data)
        teams[int(line_data[0])] = line_data[1][0:len(line_data[1]) - 1]
    
    return teams

""" 
Loads the match data of an FTC event into a list from an external txt file.
Assumes the format of each line in the txt file is:
red1, red2, redscore, redauto, blue1, blue2, bluescore, blueauto
"""
def loadMatches(filename):
    matches = []
    
    f = open(filename, 'r')
    
    matchNum = 1
    for line in f:
        line_data = line.split(' ')
        red1 = int(line_data[0])
        red2 = int(line_data[1])
        redscore = int(line_data[2])
        redauto = int(line_data[3])
        redAlliance = Alliance(red1, red2, redscore, redauto, "Red")
        
        blue1 = int(line_data[4])
        blue2 = int(line_data[5])
        bluescore = int(line_data[6])
        blueauto = int(line_data[7])
        blueAlliance = Alliance(blue1, blue2, bluescore, blueauto, "Blue")
        
        matches.append(Match(matchNum, redAlliance, blueAlliance))
        matchNum += 1
    
    return matches

# Converts any stat represented by a matrix into a list, used later for sorting
def convertToList(statMatrix):
    l = []
    for val in statMatrix:
        l.append(round(float(val), 3))
    return l

def calcs(match):
    # Load teams and matches from txt files
    teams = loadTeams("teams-"+match+".txt")
    matches = loadMatches("matches-"+match+".txt")

    """ 
    Build M, a matrix of alliances x teams, where each row indicates the teams in that alliance.
    A value of 1 means the team was in that alliance and a value of 0 means the team was not.
    First loop through each red alliance and then loop through each blue alliance.
    The resulting matrix should have 2 * len(matches) rows.
    """
    M = []
    for match in matches:
        r = []
        for team in teams:
            if match.redAlliance.team1 == team or match.redAlliance.team2 == team:
                r.append(1)
            else:
                r.append(0)
        M.append(r)

        b = []
        for team in teams:
            if match.blueAlliance.team1 == team or match.blueAlliance.team2 == team:
                b.append(1)
            else:
                b.append(0)
        M.append(b)

    """
    Build Scores, a matrix of alliances x 1, where each row indicates the score of that alliance.
    Build Autos, a matrix of alliances x 1, where each row indicates the autonomous score of that alliance.
    Build Margins, a matrix of alliances x 1, where each row indicates the margin of victory/loss 
    of that alliance (e.g. if an alliance wins 60-50, the value is +10).
    The alliance represented by each row corresponds to the alliance represented by each row
    in the matrix M.
    """
    Scores = []
    Autos = []
    Margins = []
    for match in matches:
        Scores.append([match.redAlliance.score])
        Scores.append([match.blueAlliance.score])
        Autos.append([match.redAlliance.auto])
        Autos.append([match.blueAlliance.auto])
        Margins.append([match.redAlliance.score - match.blueAlliance.score])
        Margins.append([match.blueAlliance.score - match.redAlliance.score])


    # Convert all matrices from type list to type matrix using numpy
    M = numpy.matrix(M)
    Scores = numpy.matrix(Scores)
    Autos = numpy.matrix(Autos)
    Margins = numpy.matrix(Margins)

    """ 
    Find the pseudoinverse of the matrix M. Multiplying this by a results matrix will find the
    solution to the overdetermined system of equations.
    """
    pseudoinverse = numpy.linalg.pinv(M)
    OPRs = numpy.matmul(pseudoinverse, Scores)
    AUTOs = numpy.matmul(pseudoinverse, Autos)
    CCWMs = numpy.matmul(pseudoinverse, Margins)
    
    """ Sort the teams by OPR. Created sortedTeams, sortedOPR, and sortedCCWM accordingly. """
    teamsList = [] # unsorted list of teams
    sortedTeams = []
    sortedOPR = []
    sortedAuto = []
    sortedCCWM = []

    for team in teams:
        teamsList.append(team)

    while len(sortedTeams) < len(teamsList):
        oprs = convertToList(OPRs)
        autos = convertToList(AUTOs)
        ccwms = convertToList(CCWMs)

        # Get the first team not already sorted to compare all the other teams to that team
        for i in range(len(teamsList)):
            if teamsList[i] not in sortedTeams:
                bestTeam = teamsList[i]
                bestOPR = oprs[i]
                bestAUTO = autos[i]
                bestCCWM = ccwms[i]
                break

        # Loop through teamsList to find next best team
        for i in range(len(teamsList)):
            if oprs[i] > bestOPR and teamsList[i] not in sortedTeams:
                bestTeam = teamsList[i]
                bestOPR = oprs[i]
                bestAUTO = autos[i]
                bestCCWM = ccwms[i]
        sortedTeams.append(bestTeam)
        sortedOPR.append(bestOPR)
        sortedAuto.append(bestAUTO)
        sortedCCWM.append(bestCCWM)
        
    ret = []
    
    """ Print the data to the screen """
#     print("\nTEAM\t\tOPR\t\tAuto\t\tCCWM\t\tTeam Name")
    for i in range(len(teamsList)):
        teamNum = sortedTeams[i]
        d = {}
        d['TeamNum'] = teamNum
        d['OPR'] = sortedOPR[i]
        d['Auto'] = sortedAuto[i]
        d['CCWM'] = sortedCCWM[i]
        d['TeamName'] = teams[teamNum]
        
        ret.append(d)
    print(ret)
        
    return ret


In [12]:
from csv import writer

In [13]:
base = 'https://ftc-events.firstinspires.org/2022/'

In [46]:
base2 = 'https://ftc-events.firstinspires.org'
def getQuals(match_link):#base2 + match_link+'/qualifications'
    soup = BeautifulSoup(requests.get(match_link).content, 'html.parser')
    
    quals = soup.find('tbody', {'id' : 'match-results'}).find_all('a', string = re.compile('Qualification .*'))
    
    return quals

In [47]:
def getTeams(match_link):
    soup = BeautifulSoup(requests.get(match_link.replace('/qualifications', '')).content, 'html.parser')
    
#     print(soup)
    
    table = soup.find_all('tr', {'class': ""})
    
    tl = set()
    
    for elem in table:
        e =elem.find('a', string = re.compile('\d*'))
        try:
            tl.add(e.text.strip())
        except Exception as e:
            continue
    return tl


In [48]:
def getTeamsQual(href):
    soup = BeautifulSoup(requests.get(href).content, 'html.parser')
    s = soup.find_all('div', {'class':"col-xs-6 pp-teams"})
    
    return[[s[0].find('a').text.strip(), s[1].find('a').text.strip()], [s[2].find('a').text.strip(), s[3].find('a').text.strip()]]

In [49]:
def getScores(href):
    soup = BeautifulSoup(requests.get(href).content, 'html.parser')
    bs = soup.find('div', {'class':"col-xs-4 col-md-2 pp-scoreboard-totalpoints-blue"}).text.strip()
    ba = soup.find_all('div', {'class':"pp-results-score-left col-xs-2"})[0].text.strip()
    rs = soup.find('div', {'class':"col-xs-4 col-md-2 pp-scoreboard-totalpoints-red"}).text.strip()
    ra = soup.find_all('div', {'class':"pp-results-score-right col-xs-2"})[0].text.strip()
    
    
    return [[bs, ba], [rs, ra]]

In [14]:
matches

,Link,Name,Location,Start,End
0,USARLCMP,Adventist Robotics League Championship,USA,2023-05-07 00:00:00,2023-05-07 00:00:00
1,USMIAAOS,Girls R GRAYT All Girls Event,USA,2023-05-05 00:00:00,2023-05-06 00:00:00
2,USMIGIOS,Gibraltar FTC Spring Showdown,USA,2023-05-05 00:00:00,2023-05-06 00:00:00
3,USPAPHOS,PA FTC RoboJawn,USA,2023-05-06 00:00:00,2023-05-06 00:00:00
4,FRLEOS,FR Le Défi Robotique – Lyon 2023,France,2023-05-13 00:00:00,2023-05-13 00:00:00
...,...,...,...,...,...
1135,USTXSOBEOS,FIRST in Texas UIL State 5A,USA,2023-03-24 00:00:00,2023-03-24 00:00:00
1136,CNDIS,Dianjian Peony Cup,China,2023-04-01 00:00:00,2023-04-02 00:00:00
1137,GBREQ,GB Remote Regional,United Kingdom,2023-03-23 00:00:00,2023-03-29 00:00:00
1138,USTXCECCS,TX-Central Buc Days 2023 Robot Rodeo,USA,2023-05-13 00:00:00,2023-05-13 00:00:00


In [32]:
# 0 to 10
for i in range(matches.shape[0]):
    print("i "+str(i))
    row = matches.iloc[i]
    link = row['Link']

    try:
        tset = getTeams(link)    
    
        match_list = getQuals(link)
    except Exception as e:
        continue
    
    if len(tset)==0 or len(match_list)==0:
        continue
    
    with open("./matches-"+row['Link']+".txt", "a") as mfile:
        for i in range(len(match_list)):
            curr_m = match_list[i]['href']
            team_list = getTeamsQual(curr_m)

            red = team_list[1]
            blue = team_list[0]

            res = getScores(curr_m)

            #match file
            mfile.write(str(red[0]) + " " + str(red[1]) + " " + str(res[1][0]) + " "+ str(res[1][1])+ " "+
                        str(blue[0]) + " "+ str(blue[1]) + " " + str(res[0][0]) + " " + str(res[0][1])+"\n"
                       )
    with open("./teams-"+row['Link']+".txt","a") as tfile:
        for t in tset:
            tfile.write(str(t) + ", " + teams[teams['Team Number']==int(t)].iloc[0,1]+ "\n")
        
    
    
    print(tset)
    
    results = calcs(row['Link'])
    print(row['Link'])
    
    print(results)
    
    for team_result in results:
        yes = False
        if not os.path.exists(os.getcwd().replace("scripts", "data/team_data/"+str(team_result['TeamNum'])+".csv")):
            yes = True
        with open(os.getcwd().replace("scripts", "data/team_data/"+str(team_result['TeamNum'])+".csv"), 'a') as f_object:
            writer_object = writer(f_object)
            if yes:
                writer_object.writerow(['ID', 'OPR', 'Auto', 'CCWM'])
        
                
            writer_object.writerow([row['Link'], team_result['OPR'], team_result['Auto'], team_result['CCWM']])

        f_object.close()

    
    tfile.close()
    mfile.close()
#     rmdir("teams-"+match+".txt")
#     rmdir("matches-"+match+".txt")
    

i 799
{'16917', '20150', '14931', '22110', '14521', '11409', '22161', '22162', '17178', '19706', '22178', '11873', '17008', '19542', '13546', '9617', '14561', '10075'}
[{'TeamNum': 22161, 'OPR': 98.998, 'Auto': 8.425, 'CCWM': 57.032, 'TeamName': 'M-TECH Dark Matter'}, {'TeamNum': 19706, 'OPR': 85.952, 'Auto': 16.623, 'CCWM': 46.101, 'TeamName': 'Potential Energy'}, {'TeamNum': 11873, 'OPR': 84.465, 'Auto': 28.55, 'CCWM': 57.282, 'TeamName': 'Element Unknown'}, {'TeamNum': 16917, 'OPR': 83.585, 'Auto': 6.988, 'CCWM': 39.162, 'TeamName': 'Gear Wizards'}, {'TeamNum': 22162, 'OPR': 81.406, 'Auto': 25.593, 'CCWM': 33.946, 'TeamName': 'M-TECH ProtoPlazma'}, {'TeamNum': 19542, 'OPR': 44.606, 'Auto': 7.858, 'CCWM': 6.257, 'TeamName': 'The CIA'}, {'TeamNum': 10075, 'OPR': 40.274, 'Auto': 5.606, 'CCWM': -7.293, 'TeamName': 'Tech Turtles'}, {'TeamNum': 22178, 'OPR': 39.676, 'Auto': 1.257, 'CCWM': -12.095, 'TeamName': 'Infrared'}, {'TeamNum': 14561, 'OPR': 37.111, 'Auto': 7.308, 'CCWM': -12.167, '

i 802
{'5227', '16309', '11102', '21806', '13498', '18775', '21858', '19815', '10127', '15583', '9960', '3409'}
[{'TeamNum': 5227, 'OPR': 77.878, 'Auto': 20.284, 'CCWM': 44.353, 'TeamName': 'Galactech'}, {'TeamNum': 16309, 'OPR': 73.512, 'Auto': 22.594, 'CCWM': 27.845, 'TeamName': 'Stealth Panther Robotics FTC '}, {'TeamNum': 21858, 'OPR': 70.674, 'Auto': 10.79, 'CCWM': 35.429, 'TeamName': 'Red Shift Rovers'}, {'TeamNum': 11102, 'OPR': 69.741, 'Auto': 13.919, 'CCWM': 34.318, 'TeamName': 'Team M.O.R.E. (Marshall Owls Robotics & Engineering)'}, {'TeamNum': 3409, 'OPR': 53.099, 'Auto': 2.425, 'CCWM': 4.014, 'TeamName': 'Astromechs'}, {'TeamNum': 10127, 'OPR': 47.436, 'Auto': 9.518, 'CCWM': -11.293, 'TeamName': "Tesla's Knights"}, {'TeamNum': 9960, 'OPR': 43.285, 'Auto': 7.203, 'CCWM': 2.004, 'TeamName': 'N.E.R.D.S'}, {'TeamNum': 19815, 'OPR': 42.997, 'Auto': 17.365, 'CCWM': -0.274, 'TeamName': 'The Law'}, {'TeamNum': 13498, 'OPR': 32.663, 'Auto': -1.049, 'CCWM': -14.24, 'TeamName': 'Robo 

{'17637', '8895', '8905', '9027', '5404', '7525', '6285', '11539', '14186', '4982', '14257', '5703', '13051'}
[{'TeamNum': 14186, 'OPR': 73.007, 'Auto': 19.176, 'CCWM': 63.036, 'TeamName': 'P3'}, {'TeamNum': 7525, 'OPR': 66.547, 'Auto': 16.531, 'CCWM': 53.501, 'TeamName': 'Nerdy Birds'}, {'TeamNum': 8895, 'OPR': 36.223, 'Auto': 16.489, 'CCWM': 27.215, 'TeamName': 'Loose Screws'}, {'TeamNum': 4982, 'OPR': 29.43, 'Auto': -0.001, 'CCWM': 28.422, 'TeamName': 'Cafe Bot'}, {'TeamNum': 14257, 'OPR': 28.675, 'Auto': 5.048, 'CCWM': 26.373, 'TeamName': 'REBELBOTS'}, {'TeamNum': 9027, 'OPR': 18.8, 'Auto': 2.862, 'CCWM': -7.029, 'TeamName': 'Synergistic Effect'}, {'TeamNum': 11539, 'OPR': 12.17, 'Auto': -0.891, 'CCWM': -25.231, 'TeamName': 'Team Voltage'}, {'TeamNum': 8905, 'OPR': 11.096, 'Auto': 4.105, 'CCWM': -20.162, 'TeamName': 'Statesbots- Orange'}, {'TeamNum': 5404, 'OPR': 10.868, 'Auto': 1.319, 'CCWM': -27.016, 'TeamName': 'Titanium Titans'}, {'TeamNum': 5703, 'OPR': 10.775, 'Auto': 7.631, 

i 809
{'19921', '21501', '21734', '5270', '11528', '22733', '8569', '21724', '12010', '15680', '21836', '12828', '9076', '20434', '3916', '16447', '7083', '3587', '9581', '13735', '11588', '22377', '2827', '2901'}
[{'TeamNum': 22377, 'OPR': 138.684, 'Auto': 42.751, 'CCWM': 114.995, 'TeamName': 'SigmaCorns'}, {'TeamNum': 11528, 'OPR': 88.498, 'Auto': 17.408, 'CCWM': 52.224, 'TeamName': 'Bots of Prey'}, {'TeamNum': 20434, 'OPR': 74.27, 'Auto': 17.938, 'CCWM': 51.098, 'TeamName': 'Binding Energy'}, {'TeamNum': 12828, 'OPR': 70.884, 'Auto': 22.282, 'CCWM': 46.577, 'TeamName': 'Critical Overload'}, {'TeamNum': 19921, 'OPR': 60.772, 'Auto': 17.195, 'CCWM': 37.556, 'TeamName': 'Space Invaders'}, {'TeamNum': 21501, 'OPR': 58.358, 'Auto': 6.393, 'CCWM': 46.937, 'TeamName': 'Ohms'}, {'TeamNum': 7083, 'OPR': 58.085, 'Auto': 19.216, 'CCWM': 13.26, 'TeamName': 'TundraBots'}, {'TeamNum': 3587, 'OPR': 53.18, 'Auto': 8.94, 'CCWM': 54.64, 'TeamName': 'Unparalleled Processing'}, {'TeamNum': 13735, 'OPR'

i 811
{'8888', '21689', '22879', '12622', '7135', '19695', '14590', '20910', '22345', '17630', '13738', '11536', '15343', '15534', '12758', '17455', '22880', '19643', '22346', '20099'}
[{'TeamNum': 12758, 'OPR': 89.029, 'Auto': 18.219, 'CCWM': 67.341, 'TeamName': 'Derryfield Binary Bots'}, {'TeamNum': 15343, 'OPR': 86.801, 'Auto': 23.147, 'CCWM': 18.904, 'TeamName': 'Blue VIII'}, {'TeamNum': 15534, 'OPR': 65.322, 'Auto': 10.043, 'CCWM': 49.912, 'TeamName': 'VERTEX'}, {'TeamNum': 19695, 'OPR': 61.458, 'Auto': 27.723, 'CCWM': 32.589, 'TeamName': 'Flying Screwdrivers'}, {'TeamNum': 11536, 'OPR': 60.7, 'Auto': 19.371, 'CCWM': 35.812, 'TeamName': 'Derryfield Fighting Cougars'}, {'TeamNum': 19643, 'OPR': 55.335, 'Auto': 18.024, 'CCWM': 18.051, 'TeamName': 'Dragon Droids'}, {'TeamNum': 17455, 'OPR': 44.47, 'Auto': 16.355, 'CCWM': -12.634, 'TeamName': 'The ... Cephalo-Bots?'}, {'TeamNum': 20910, 'OPR': 35.685, 'Auto': 6.427, 'CCWM': 21.384, 'TeamName': 'Hornets 2'}, {'TeamNum': 17630, 'OPR': 3

...

i 1125
{'19927', '20293', '9225', '19367', '19796', '11570', '14631', '4106', '6914', '10100', '16310', '21923', '22548', '7974', '21355', '8680', '9956', '10824', '19417', '13201', '16460', '21180', '4115', '17235', '10829'}
[{'TeamNum': 8680, 'OPR': 151.067, 'Auto': 44.448, 'CCWM': 80.511, 'TeamName': 'Kraken-Pinion '}, {'TeamNum': 21180, 'OPR': 122.287, 'Auto': 35.497, 'CCWM': 38.787, 'TeamName': 'Error 418'}, {'TeamNum': 9956, 'OPR': 114.151, 'Auto': 46.466, 'CCWM': 26.571, 'TeamName': 'The Knack'}, {'TeamNum': 9225, 'OPR': 112.434, 'Auto': 30.947, 'CCWM': 36.674, 'TeamName': 'Dynamite Social Club'}, {'TeamNum': 10100, 'OPR': 106.053, 'Auto': 34.297, 'CCWM': 56.84, 'TeamName': 'Phoenix Force'}, {'TeamNum': 16310, 'OPR': 104.464, 'Auto': 30.218, 'CCWM': 17.376, 'TeamName': 'Rising Rhinobots'}, {'TeamNum': 16460, 'OPR': 103.143, 'Auto': 34.478, 'CCWM': 32.143, 'TeamName': 'GEarheads'}, {'TeamNum': 14631, 'OPR': 95.34, 'Auto': 17.26, 'CCWM': 38.967, 'TeamName': 'Laser Tech'}, {'TeamNu

i 1127
i 1128
i 1129
{'20999', '21003', '19282', '21207', '19279', '21187', '19284', '22640', '20994', '21206', '20997', '22456', '22877', '21200', '21186', '22457', '21199', '23053', '20998', '21122', '21009', '21002', '19280', '20992'}
[{'TeamNum': 19280, 'OPR': 50.475, 'Auto': 5.325, 'CCWM': 32.68, 'TeamName': 'UK 235 - Blackout Robotics '}, {'TeamNum': 22640, 'OPR': 40.402, 'Auto': 10.258, 'CCWM': 33.156, 'TeamName': 'UK 465 - Harrow School - Team Euclid'}, {'TeamNum': 21207, 'OPR': 27.632, 'Auto': -1.433, 'CCWM': 12.112, 'TeamName': 'TEAM 425'}, {'TeamNum': 20997, 'OPR': 21.372, 'Auto': 5.251, 'CCWM': 8.946, 'TeamName': 'fernDOWNLOAD#166'}, {'TeamNum': 19284, 'OPR': 20.364, 'Auto': 2.59, 'CCWM': 3.112, 'TeamName': 'UK 248 - ASL Team 5 - MS ASP G8'}, {'TeamNum': 20994, 'OPR': 20.194, 'Auto': 1.057, 'CCWM': -12.486, 'TeamName': 'fernDOWNLOAD#85'}, {'TeamNum': 22457, 'OPR': 16.456, 'Auto': 13.549, 'CCWM': 0.865, 'TeamName': 'UK 456 - BSG - BSGRobotics'}, {'TeamNum': 20999, 'OPR': 16.

i 1132
{'10528', '11986', '21856', '10862', '11720', '12115', '15930', '15546', '10405', '16617', '20410', '13109', '10676', '16215', '12398', '18483', '5175', '16756', '13472', '15877', '9778', '18458', '21899', '21733'}
[{'TeamNum': 15546, 'OPR': 125.897, 'Auto': 27.462, 'CCWM': 78.305, 'TeamName': 'Monkey Mechanics'}, {'TeamNum': 10862, 'OPR': 113.068, 'Auto': 35.827, 'CCWM': 45.643, 'TeamName': 'Droid Rage - Nebula'}, {'TeamNum': 16617, 'OPR': 103.108, 'Auto': 24.149, 'CCWM': 26.647, 'TeamName': 'Nazareth Robotics'}, {'TeamNum': 12398, 'OPR': 101.565, 'Auto': 13.142, 'CCWM': 58.272, 'TeamName': 'Bobcat Robotics'}, {'TeamNum': 18483, 'OPR': 97.502, 'Auto': 23.283, 'CCWM': 38.018, 'TeamName': 'Buckar00s'}, {'TeamNum': 21733, 'OPR': 97.349, 'Auto': 18.42, 'CCWM': 35.315, 'TeamName': 'Kryptic Knights'}, {'TeamNum': 18458, 'OPR': 87.908, 'Auto': 16.333, 'CCWM': 17.964, 'TeamName': 'Battery Chargerz'}, {'TeamNum': 11986, 'OPR': 83.98, 'Auto': 13.535, 'CCWM': 20.499, 'TeamName': 'Code Blu

i 1134
i 1135
{'17802', '8284', '13590', '14361', '13227', '12448', '14637', '17294', '20325', '4921', '6832', '20736', '13539', '16320', '20313', '8811', '12209', '19857', '13265', '12126', '13825', '4902', '15725'}
[{'TeamNum': 14361, 'OPR': 137.432, 'Auto': 34.222, 'CCWM': 85.798, 'TeamName': 'CPHS ROBOLOBOS GREEN'}, {'TeamNum': 8284, 'OPR': 117.679, 'Auto': 36.226, 'CCWM': 47.34, 'TeamName': 'Soap Dispenser'}, {'TeamNum': 4921, 'OPR': 111.864, 'Auto': 13.457, 'CCWM': 97.738, 'TeamName': 'The Dropouts'}, {'TeamNum': 13265, 'OPR': 96.718, 'Auto': 20.419, 'CCWM': 98.721, 'TeamName': 'Osiris'}, {'TeamNum': 20325, 'OPR': 94.326, 'Auto': 17.979, 'CCWM': 39.907, 'TeamName': 'Maximum Resistance'}, {'TeamNum': 17294, 'OPR': 87.314, 'Auto': 21.115, 'CCWM': 1.48, 'TeamName': 'Over Dr!ve Robotics'}, {'TeamNum': 20313, 'OPR': 82.562, 'Auto': 24.47, 'CCWM': 23.03, 'TeamName': 'Mustang Robotics'}, {'TeamNum': 14637, 'OPR': 73.705, 'Auto': 17.636, 'CCWM': 1.12, 'TeamName': 'Aquanauts'}, {'TeamNum'

i 1137
i 1138
{'5243', '16597', '16556', '10862', '18908', '15703', '19746', '13893', '5188', '15654', '13833', '18738', '15221', '21352', '21879', '13109', '17229', '16828', '12977', '11924', '16458', '9923', '12928', '17329', '19412', '13266', '13744', '6901', '12202', '16377', '16555', '16832', '18458', '21386', '16831', '12302', '4902', '15725'}
[{'TeamNum': 19746, 'OPR': 171.953, 'Auto': 38.537, 'CCWM': 130.496, 'TeamName': 'The Disruptingly Robocephalic BrainSTEM Robotics Team'}, {'TeamNum': 16458, 'OPR': 131.028, 'Auto': 42.313, 'CCWM': 107.74, 'TeamName': 'TechnoWizards'}, {'TeamNum': 12928, 'OPR': 117.986, 'Auto': 24.696, 'CCWM': 58.089, 'TeamName': 'LightSaders'}, {'TeamNum': 16377, 'OPR': 116.897, 'Auto': 32.286, 'CCWM': 78.851, 'TeamName': 'Spicy Ketchup'}, {'TeamNum': 18908, 'OPR': 89.922, 'Auto': 13.98, 'CCWM': 59.654, 'TeamName': 'Mighty Hawks'}, {'TeamNum': 13266, 'OPR': 88.394, 'Auto': 7.17, 'CCWM': 18.993, 'TeamName': 'Droid Rage - Apex'}, {'TeamNum': 12302, 'OPR': 82

In [36]:
missing = []
for i in range(0,matches.shape[0]):
    print("i "+str(i))
    row = matches.iloc[i]
    link = row['Link']

    try:
        tset = getTeams(link)    
    
        match_list = getQuals(link)
    except Exception as e:
        continue
    
    if len(tset)==0 or len(match_list)==0:
        missing.append(i)
    
    
    

i 0
i 1
i 2
i 3
i 4
i 5
i 6
i 7
i 8
i 9
i 10
i 11
i 12
i 13
i 14
i 15
i 16
i 17
i 18
i 19
i 20
i 21
i 22
i 23
i 24
i 25
i 26
i 27
i 28
i 29
i 30
i 31
i 32
i 33
i 34
i 35
i 36
i 37
i 38
i 39
i 40
i 41
i 42
i 43
i 44
i 45
i 46
i 47
i 48
i 49
i 50
i 51
i 52
i 53
i 54
i 55
i 56
i 57
i 58
i 59
i 60
i 61
i 62
i 63
i 64
i 65
i 66
i 67
i 68
i 69
i 70
i 71
i 72
i 73
i 74
i 75
i 76
i 77
i 78
i 79
i 80
i 81
i 82
i 83
i 84
i 85
i 86
i 87
i 88
i 89
i 90
i 91
i 92
i 93
i 94
i 95
i 96
i 97
i 98
i 99
i 100
i 101
i 102
i 103
i 104
i 105
i 106
i 107
i 108
i 109
i 110
i 111
i 112
i 113
i 114
i 115
i 116
i 117
i 118
i 119
i 120
i 121
i 122
i 123
i 124
i 125
i 126
i 127
i 128
i 129
i 130
i 131
i 132
i 133
i 134
i 135
i 136
i 137
i 138
i 139
i 140
i 141
i 142
i 143
i 144
i 145
i 146
i 147
i 148
i 149
i 150
i 151
i 152
i 153
i 154
i 155
i 156
i 157
i 158
i 159
i 160
i 161
i 162
i 163
i 164
i 165
i 166
i 167
i 168
i 169
i 170
i 171
i 172
i 173
i 174
i 175
i 176
i 177
i 178
i 179
i 180
i 181
i 182
i 183
i 184


In [37]:
df = pd.DataFrame(missing)
df

,0
0,3
1,4
2,9
3,11
4,13
...,...
122,1128
123,1131
124,1134
125,1137


In [4]:
missing = pd.read_csv(os.getcwd().replace('scripts', 'data') + '/missing.csv')

In [6]:
missing

,0
0,3
1,4
2,9
3,11
4,13
...,...
122,1128
123,1131
124,1134
125,1137


In [8]:
for i in range(127):
    print(str(i)+" "+matches.iat[missing.iat[i,0],0]+" "+matches.iat[missing.iat[i,0],1])
    
    

0 USPAPHOS PA FTC RoboJawn
1 FRLEOS FR Le Défi Robotique – Lyon 2023
2 USORROOS OR Spring Showcase
3 USAZFLOS AZ Lunar Legacy Invitational
4 USORGLOS OR Spring Showcase 2
5 USORMLS OR Spring Scrimmage and Showcase
6 GBCAOS GB New Teams Regional
7 GBCMP Championship- United Kingdom
8 USALHUOS AL betaSEIT
9 USOKMUOS OK-Northeast (NEO) Invitational
10 USCANOSMOS Lobster Cup International
11 AUMPOS1 AU APOC 2023
12 USILCHOS IL Chicago Robotics Invitational
13 USOHAKOS OH KSS Robotics Turtle Friendly
14 USNDBIS ND Scrimmage
15 USAZTUS AZ Tucson "Meet the Field" Day
16 USMNSPS MN Cass Lake League Meet 0
17 USMSBCS1 MS Bogue Chitto Build-it Day/Scrimmage
18 USIDPFS1 ID North Idaho Scrimmage #1
19 USAKJUS2 AK Southeast Scrimmage #1
20 USFLFLS FL South Florida League Scrimmage and Volunteer Training
21 USGAGAS GA N Georgia League Scrimmage
22 USMNAVS MN FTC Mille Lacs / Pepin League Meet 0
23 USMSLAS MS LCS Build-it Day/Scrimmage
24 USORRBS OR North Coast League Meet 0
25 USTXWPMIS TX-W&P Midla

In [25]:
ids = [ 'USTXCECMP','FTCCMP1', 'USNJCMP', 'USMICMP2', 'USMICMP', 'CAABABSALT','ILCMP','USCHSCMP','USMNCMP','USAZCMP','USCANOCMP','USFLCMP','USIACMP']



In [28]:
from selenium.webdriver.common.by import By

In [59]:
# 0 to 10

driver = webdriver.Safari()

for i in range(2,len(ids)):
    print("i "+str(i))
    link = ids[i]
    
    driver.get('https://ftc-events.firstinspires.org/2022/'+link)
    
    subs = driver.find_elements(By.CLASS_NAME, 'col-sm-4')
    
    print(subs)
    
    if len(subs) > 0 and len(subs)%3==0:
        for j in range(len(subs)//3):
            link = subs[j*3].find_element(By.CLASS_NAME, "btn").get_attribute('href')
            
            print(link)

            try:
                tset = getTeams(link)    

                match_list = getQuals(link)
            except Exception as e:
                continue
                
            print(tset)

            if len(tset)==0 or len(match_list)==0:
                continue

            with open("./matches-"+link.split('/')[-2]+".txt", "a") as mfile:
                for i in range(len(match_list)):
                    curr_m = match_list[i]['href']
                    team_list = getTeamsQual('https://ftc-events.firstinspires.org'+curr_m)

                    red = team_list[1]
                    blue = team_list[0]

                    res = getScores('https://ftc-events.firstinspires.org'+curr_m)

                    #match file
                    mfile.write(str(red[0]) + " " + str(red[1]) + " " + str(res[1][0]) + " "+ str(res[1][1])+ " "+
                                str(blue[0]) + " "+ str(blue[1]) + " " + str(res[0][0]) + " " + str(res[0][1])+"\n"
                               )
            with open("./teams-"+link.split('/')[-2]+".txt","a") as tfile:
                for t in tset:
                    tfile.write(str(t) + ", " + teams[teams['Team Number']==int(t)].iloc[0,1]+ "\n")



            print(tset)

            results = calcs(link.split('/')[-2])
#             print(row['Link'])

            print(results)

            for team_result in results:
                yes = False
                if not os.path.exists(os.getcwd().replace("scripts", "data/team_data/"+str(team_result['TeamNum'])+".csv")):
                    yes = True
                with open(os.getcwd().replace("scripts", "data/team_data/"+str(team_result['TeamNum'])+".csv"), 'a') as f_object:
                    writer_object = writer(f_object)
                    if yes:
                        writer_object.writerow(['ID', 'OPR', 'Auto', 'CCWM'])


                    writer_object.writerow([link.split('/')[-2], team_result['OPR'], team_result['Auto'], team_result['CCWM']])

                f_object.close()


            tfile.close()
            mfile.close()
#     rmdir("teams-"+match+".txt")
#     rmdir("matches-"+match+".txt")
    

i 2
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-3C999936-3990-45D9-9437-FDA284D2E0BE")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-7C24678C-DD6A-4DC0-8684-C65B79C1300A")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-4159548C-24D8-46B8-94E4-5D4E9795E1E2")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-F4D54820-1016-4359-B685-46E8FA7FD3B9")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-3BA8E2DD-339B-4540-8B7E-31D18C61461C")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-37115203-062E-4373-AE5B-584190B07791")>]
https://ftc-events.firstinspires.org/2022/USNJCMPTPKE/qualifications
{'10096'

i 3
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-34E604F4-C7E1-40C4-A26E-8DA02D0F5B21")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-38753090-585C-4B04-AC91-3B452E1C4F35")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-13DE93C2-BCD2-420A-BAFC-ED9DADA58258")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-B085ABCD-AD2A-4344-8A7C-D3EAA7F6E291")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-8303D6B5-3700-4B06-BF79-9A322AB97425")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-FE65C3C8-8E3E-4A7F-A821-0120C97F7281")>]
https://ftc-events.firstinspires.org/2022/USMICMP2WF/qualifications
{'5957', 

https://ftc-events.firstinspires.org/2022/USMICMP2MC/qualifications
{'5291', '8487', '15004', '8579', '19949', '17993', '21791', '9819', '16883', '10136', '14815', '10016', '13019', '14757', '16536', '7215', '14670', '8531', '11254', '19923', '17181', '10698', '15638', '21363', '14638', '8777', '8845', '15256', '21331', '18954', '7043', '14010', '7252', '22308', '13999', '18474'}
{'5291', '8487', '15004', '8579', '19949', '17993', '21791', '9819', '16883', '10136', '14815', '10016', '13019', '14757', '16536', '7215', '14670', '8531', '11254', '19923', '17181', '10698', '15638', '21363', '14638', '8777', '8845', '15256', '21331', '18954', '7043', '14010', '7252', '22308', '13999', '18474'}
[{'TeamNum': 15004, 'OPR': 126.086, 'Auto': 33.461, 'CCWM': 45.525, 'TeamName': 'Tech KNOW Logic'}, {'TeamNum': 7252, 'OPR': 109.499, 'Auto': 34.127, 'CCWM': 27.198, 'TeamName': 'Vicious Volts'}, {'TeamNum': 10016, 'OPR': 101.935, 'Auto': 22.867, 'CCWM': 37.369, 'TeamName': 'Clarkston RoboWolves'}, {'

{'8027', '13702', '11244', '21906', '21672', '5384', '15301', '6494', '16607', '10024', '5292', '15465', '5383', '10477', '10031', '6155', '10062', '22313', '15270', '15182', '10723', '7643', '21453', '10309', '10644', '8646', '15815', '19561', '5386', '14649', '7031', '12845', '19925', '5618', '11266', '10601'}
{'8027', '13702', '11244', '21906', '21672', '5384', '15301', '6494', '16607', '10024', '5292', '15465', '5383', '10477', '10031', '6155', '10062', '22313', '15270', '15182', '10723', '7643', '21453', '10309', '10644', '8646', '15815', '19561', '5386', '14649', '7031', '12845', '19925', '5618', '11266', '10601'}
[{'TeamNum': 19561, 'OPR': 141.112, 'Auto': 39.682, 'CCWM': 32.77, 'TeamName': 'Tiger Tech Black'}, {'TeamNum': 10644, 'OPR': 128.805, 'Auto': 37.334, 'CCWM': 62.614, 'TeamName': 'CyBugs'}, {'TeamNum': 7643, 'OPR': 126.847, 'Auto': 43.284, 'CCWM': 68.057, 'TeamName': 'Tiger Tech Orange'}, {'TeamNum': 15465, 'OPR': 110.325, 'Auto': 27.825, 'CCWM': 33.644, 'TeamName': 'Te

i 5
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-C825AEA4-272C-4ACF-B04B-3F71DA340574")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-2104D76F-55AA-4419-90A7-2AB6CF14C3B5")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-43ADA6CF-90E1-45B3-99D4-397DF1AB8980")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-21714428-A622-47DD-9AC2-BE9BFE36B56F")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-E4C7665A-F7EE-4F1F-93D5-79A958194DBD")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-539E6A0F-4273-4F69-8C98-EE3C7BF593DC")>]
https://ftc-events.firstinspires.org/2022/CAABABSALTFIN/qualifications
{'1001

...

i 8
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-84677345-5ACC-450B-A6FB-DBD84A175B9A")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-97E18844-91DE-4084-B4E0-6F1766C36B9C")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-B9B1CEF7-A486-42C9-8D20-E969A06EFCF9")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-7BCA7B59-16DA-4EF1-A0A7-CC76F9B7CE5A")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-0B6E2369-3BA1-4EC4-B71F-33FAC4486507")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-FB7E12E6-B360-4E5C-ACE3-2529247B2666")>]
https://ftc-events.firstinspires.org/2022/USMNCMPGAL1/qualifications
{'10470'

i 9
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-759F91AC-70CE-4F40-BE37-F7AA924DBB84")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-2485D978-0CAA-406D-BD3F-125495746330")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-CABC7B72-4770-445F-B6E9-5DF90E331252")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-7975E438-5A94-48BE-A739-697E7C34928A")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-B6FFDB06-3703-4DA4-8A0A-9D4EC816D1A8")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-9928DF63-CBF3-4D02-BB53-B7C9E6DC00C3")>]
https://ftc-events.firstinspires.org/2022/USAZCMPGC23/qualifications
{'6174',

i 10
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-074855A9-1A20-47E8-A875-2C642C8DDCFC")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-E2F33EA9-F2C4-4AEF-98AD-EC3F259A4574")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-F7713FE2-08A1-4371-84D6-7FBA2450933B")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-E2252069-FD0E-4BA2-B60E-8450392E1737")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-1412975C-6D56-435A-B17B-2FBB592519B9")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-1740C684-6FB6-4950-8EE5-A98C9574F4E6")>]
https://ftc-events.firstinspires.org/2022/USCANOCMPAU/qualifications
{'16532

i 11
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-93F77BFD-B01F-446A-BDB2-DB23359ACF2D")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-2823E46C-6FF9-4AC7-B7A4-8144436B38A2")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-D79F0560-196E-4E54-94E6-3696ADFD93F3")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-62D5DA42-7731-4AFB-BD2F-953596101C83")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-AC560D0C-6819-4871-B041-256D6026D534")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-790F7662-9F45-41AC-AC82-6AC0FD0990FA")>]
https://ftc-events.firstinspires.org/2022/USFLCMPSCOT/qualifications
{'9214'

i 12
[<selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-49E11512-C374-4F8E-96AB-8497728D012F")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-5BC05070-8706-4876-B5A6-11E4582B95C6")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-19FEFD3D-18E6-40DA-9380-6676F617C17F")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-453B99B1-9438-42C4-9C16-250DEF5E8D80")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-C5D62614-FE84-4696-BF03-04B9679C850A")>, <selenium.webdriver.remote.webelement.WebElement (session="66F8E973-5C7E-4A33-B1E2-B8BAABF61E14", element="node-6504D378-05A7-4D70-8159-46452B35F6E9")>]
https://ftc-events.firstinspires.org/2022/USIACMPGD/qualifications
{'7110', 

In [47]:
matches.iat[m.iat[124,0], 0]

'USTXCECMP'

In [48]:
missing = ['USMICMP2', 'USMICMP', 'CAABABSALT', 'ILCMP', 'USCHSCMP', 'USMNCMP', 'USAZCMP', 'USORCMPSOL', 'USORCMPHYD', 'USCANOCMP', 'USFLCMP', 'USIACMP', 'USNJCMP', 'USTXSOCMP2', 'USTXCECMP', 'FTCCMP1']



In [49]:
cmps = pd.DataFrame(columns = ['id'])

In [ ]:
cmps.to_csv(os.getcwd().replace('scripts', 'data') + '/bad.csv')

In [38]:
df.to_csv(os.getcwd().replace('scripts', 'data') + '/missing.csv', index = False)